# UAV Visual Localization Demo

This notebook demonstrates the complete workflow for **Cross-View UAV Visual Localization**:
1. **Dataset Download & Extraction:** Automatically fetching and unpacking dataset archives.
2. **Model Fine-Tuning:** Training DINOv2 + GeM pooling head on 10 epochs using Triplet Loss across sub-datasets `[1, 2, 3, 4, 5]`.
3. **Global Retrieval:** Coarse search of Top-$K$ satellite tiles using L2-normalized DINOv2 embeddings.
4. **Local Match & Geolocalization:** Keypoint matching via **SuperPoint + LightGlue** and homography pose estimation via **MAGSAC++** to project image center into GPS coordinates.
5. **Comprehensive Metrics & Evaluation:** Returning evaluation metrics as a `pandas.DataFrame` (Mean/Median Error, P90, Acc $\le 50\text{m}/100\text{m}/500\text{m}$, Top-1/5/K Recall).
6. **Visualizations:** Plotting satellite tile similarity and keypoint homography matches.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve() if Path("..").joinpath("uav_visloc").exists() else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import random
import shutil

import numpy as np
import pandas as pd
import torch
import yaml
from lightglue import LightGlue, SuperPoint
from torch.utils.data import DataLoader

from uav_visloc.dataset import UAVSingleDataset, UAVTripletDataset, download_dataset
from uav_visloc.models import (
    DINOv2_Embedder,
    evaluate_model,
    get_embeddings,
    train_model,
    visualize_refined_matches,
)
from uav_visloc.utils import get_top_similar, visualize_top_k

print("Setup completed successfully!")

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
config_path = PROJECT_ROOT / "config.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

dataset_link = config.get("dataset_link", "https://huggingface.co/datasets/nikon942/UAV-VisLoc-Folder06/resolve/main/data_06.zip")
data_folder = config["data_folder"]
raw_folder = config["raw_folder"]
extract_folder = config["extract_folder"]
zip_file_name = config["zip_file_name"]

dataset_path = PROJECT_ROOT / data_folder / raw_folder / extract_folder
destination_file = PROJECT_ROOT / data_folder / raw_folder / zip_file_name

train_folders = [1, 2, 3, 4, 5]  # Trained on datasets 1-5
test_folders = [6]               # Evaluated on dataset 6
epochs = 10                       # 10 Training epochs
batch_size = 32
tile_scales = config.get("tile_scales", [518, 1036, 1554])
top_k = int(config.get("top_k", 10))
drone_res_height = config.get("drone_res_height", 1036)
drone_res_width = config.get("drone_res_width", 518)

print(f"Dataset Path: {dataset_path}")
print(f"Train Subsets: {train_folders}")
print(f"Test Subsets: {test_folders}")
print(f"Training Epochs: {epochs}")

In [ ]:
if not dataset_path.exists() or not any(dataset_path.iterdir()):
    if not destination_file.exists():
        print(f"Downloading dataset from {dataset_link}")
        destination_file.parent.mkdir(parents=True, exist_ok=True)
        download_dataset(url=dataset_link, destination_path=destination_file)
        print("Dataset zip downloaded successfully!")
    else:
        print("Zip archive already present locally.")

    print(f"Extracting archive to {dataset_path}")
    dataset_path.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(destination_file, dataset_path)
    print("Dataset extracted successfully!")
else:
    print(f"Dataset already available at: {dataset_path}")

In [ ]:
print("Initializing DINOv2 Embedder Architecture")
model = DINOv2_Embedder(embed_dim=1024).to(device)
print(model)

In [ ]:
print("Loading UAV Triplet Training Datasets")
train_dataset = UAVTripletDataset(dataset_path=dataset_path, folder_list=train_folders)
test_dataset = UAVTripletDataset(dataset_path=dataset_path, folder_list=test_folders)

print(f"Total Training Triplets: {len(train_dataset)}")
print(f"Total Test Triplets: {len(test_dataset)}")

In [ ]:
print(f"Starting model training for {epochs} epochs")
trained_model = train_model(
    model=model,
    train_dataset=train_dataset,
    test_dataset=test_dataset,
    device=device,
    batch_size=batch_size,
    epoch=epochs
)
print("Training completed successfully!")

In [ ]:
print("Loading SuperPoint Extractor & LightGlue Matcher")
extractor = SuperPoint(max_num_keypoints=4096).eval().to(device)
matcher = LightGlue(features="superpoint").eval().to(device)

print("Running Full Evaluation & Computing Comprehensive Metrics")
metrics_df = evaluate_model(
    model=trained_model,
    extractor=extractor,
    matcher=matcher,
    top_k=top_k,
    dataset_path=dataset_path,
    test_folders=test_folders,
    scales=tile_scales,
    device=device
)

metrics_df

In [ ]:
test_folder = test_folders[0]

drone_dataset = UAVSingleDataset(dataset_path=dataset_path, folder_list=[test_folder], is_tile=False)
drone_dataloader = DataLoader(dataset=drone_dataset, batch_size=32, shuffle=False)
drone_embeddings = get_embeddings(dataloader=drone_dataloader, model=trained_model, device=device)

satelite_dataset = UAVSingleDataset(dataset_path=dataset_path, folder_list=[test_folder], is_tile=True, scales=tile_scales)
satelite_dataloader = DataLoader(dataset=satelite_dataset, batch_size=32, shuffle=False)
satelite_embeddings = get_embeddings(dataloader=satelite_dataloader, model=trained_model, device=device)

topk_scores, topk_indices = get_top_similar(
    drone_embeddings=drone_embeddings,
    satelite_embeddings=satelite_embeddings,
    top_k=top_k
)

print("Visualizing Top-K Satellite Tile Similarity Results")
visualize_top_k(
    num_samples=3,
    drone_dataset=drone_dataset,
    satelite_dataset=satelite_dataset,
    topk_indices=topk_indices,
    topk_scores=topk_scores
)

In [ ]:
print("Visualizing SuperPoint + LightGlue Keypoint Homography & Projected Center")
visualize_refined_matches(
    num_samples=3,
    extractor=extractor,
    matcher=matcher,
    drone_dataset=drone_dataset,
    satelite_dataset=satelite_dataset,
    topk_indices=topk_indices,
    device=device,
    drone_res_height=drone_res_height,
    drone_res_width=drone_res_width
)